# Visualisation & Debugging

This notebook covers the tools available for understanding and debugging
a `pygeodata` pipeline:

- Plotting the runtime execution graph
- Plotting the static class dependency graph
- Inspecting hashes and parameters
- Cleaning stale cache entries

In [ ]:
import os, sys
os.chdir('../../..')
sys.path.insert(0, 'docs/loaders')

In [ ]:
import json
from pathlib import Path

from pygeodata import SpatialSpec, clean_cache, get_config, load
from pygeodata.graphs import plot_class_dependency_graph, plot_compact_execution_graph

from pipeline import CountryMaskLoader, LandWaterTableDepth, WaterTableDepthLoader

get_config().update(path_cache=Path('data/processed'))

spec = SpatialSpec.from_raster_file('data/wtd.tif')

wtd   = WaterTableDepthLoader()
mask  = CountryMaskLoader()
root  = LandWaterTableDepth(wtd=wtd, mask=mask)

# Make sure the pipeline has run so caches exist
load(root, spec)
print('Pipeline ready.')

## 1. Runtime execution graph

`plot_compact_execution_graph` renders the *instance-level* dependency graph:
all `Data` instances wired via their parameters. Nodes show class name and
parameter values; edges are labelled with the parameter name.

Requires: `pip install graphviz` and the system `graphviz` package.

In [ ]:
plot_compact_execution_graph(root)

## 2. Static class dependency graph

`plot_class_dependency_graph` shows *class-level* relationships:
solid arrows for inheritance, dashed arrows for method-call references.
This reflects the static structure of the codebase, not a specific run.

In [ ]:
plot_class_dependency_graph(LandWaterTableDepth)

## 3. Inspect hashes and parameters

Each cached output is identified by a **state hash** combining:

- `get_dependency_tree_hash()` — SHA-256 of the AST of the class graph
- The serialised parameter values

The `instance_hash` is a spec-independent fingerprint of the loader instance
(class code + params, no spec). It is stable across specs.

In [ ]:
print('Parameters:')
for k, v in root.get_params().items():
    print(f'  {k}: {v!r}')

print()
print('Dependency tree hash:', LandWaterTableDepth.get_dependency_tree_hash())
print('Instance hash:       ', root.get_instance_hash())
print('State hash:          ', root.get_state_hash(spec))
print()
print('Output path:   ', root.get_processed_path(spec))
print('Is processed:  ', root.is_processed(spec))
print('Cache valid:   ', root.is_cache_valid(spec))

## 4. Full dependency tree

`get_dependency_tree()` returns a nested dict of call and inheritance
dependencies with their AST hashes. This is exactly what the caching system
uses to detect whether any upstream class has changed.

In [ ]:
tree = LandWaterTableDepth.get_dependency_tree()
print(json.dumps(tree, indent=2))

## 5. Clean stale cache entries

`clean_cache` removes outputs whose on-disk hash no longer matches the live
class code. Use `dry_run=True` first to see what would be deleted.

With no `loader` argument it scans the entire cache directory.

In [ ]:
# Dry run for LandWaterTableDepth only
clean_cache(loader=LandWaterTableDepth, dry_run=True)

# To apply:
# clean_cache(loader=LandWaterTableDepth, dry_run=False)

# To scan everything:
# clean_cache(dry_run=True)

## 6. Serialised parameters on disk

Every cached output has a `.params.json` file alongside it. `get_params_as_json()`
returns the same dict — useful for verifying what parameters produced a
particular file without re-running the loader.

In [ ]:
print('Serialised params for root loader:')
print(json.dumps(root.get_params_as_json(spec=spec), indent=2))